# Week 54, Day 5: DP practice

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w054d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
stones = [4, 1, 3]
prefix = [0]
for s in stones:
    prefix.append(prefix[-1] + s)
n = len(stones)
cost = [[0] * n for _ in range(n)]
for length in range(2, n + 1):
    for i in range(n - length + 1):
        j = i + length - 1
        cost[i][j] = min(cost[i][k] + cost[k + 1][j] for k in range(i, j)) + prefix[j + 1] - prefix[i]
print(cost)

In [ ]:
def chain_cost(dims):
    """dims[i], dims[i + 1]: the shape of matrix i. Fewest multiplications."""
    n = len(dims) - 1
    cost = [[0] * n for _ in range(n)]
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            cost[i][j] = min(cost[i][k] + cost[k + 1][j] + dims[i] * dims[k + 1] * dims[j + 1]
                             for k in range(i, j))
    return cost[0][n - 1]

print(chain_cost([10, 30, 5, 60]), chain_cost([40, 20, 30, 10, 30]))

In [ ]:
from functools import cache

def count_digit_sum(limit, target):
    """How many numbers from 0 to limit have digits adding up to target."""
    digits = list(map(int, str(limit)))

    @cache
    def go(pos, total, tight):
        if total > target:
            return 0
        if pos == len(digits):
            return 1 if total == target else 0
        top = digits[pos] if tight else 9
        return sum(go(pos + 1, total + d, tight and d == top) for d in range(top + 1))

    return go(0, 0, True)

print(count_digit_sum(100, 10), count_digit_sum(10 ** 18, 9))

In [ ]:
def fewest_drops(eggs, floors):
    reach = [0] * (eggs + 1)              # reach[k]: floors handled with t drops and k eggs
    t = 0
    while reach[eggs] < floors:
        t += 1
        for k in range(eggs, 0, -1):
            reach[k] = reach[k - 1] + 1 + reach[k]
    return t

print(fewest_drops(1, 10), fewest_drops(2, 100), fewest_drops(3, 10 ** 6))

> **Key idea.** Go from problem to DP with a checklist: the function and its smallest arguments, the last decision, base cases and the answer, a fill order, states × work, and reconstruction. Interval DPs use f(i, j) over a stretch, choose a split point k, fill by increasing length, and cost O(n³). Digit DPs count numbers up to a huge limit with the state (position, a small fact, tight). When a state is too large, swap what you're given with what you compute, as in the egg drop's floors(t, k).

### Exercise: Merging piles

Neighbouring piles of stones are merged two at a time until one pile remains; each merge costs the size of the new pile. Write `merge_cost(piles)`: the least total cost, by interval DP with prefix sums. Then `merge_plan(piles)`: the best bracketing as a string, using the piles' sizes and brackets, like `"(4 (1 3))"`, keeping the **first** best split point; a single pile is just its number.

In [ ]:
def merge_cost(piles):
    ___

def merge_plan(piles):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom functools import cache\ndef brute(piles):\n    @cache\n    def go(t):\n        if len(t) == 1:\n            return 0\n        return min(go(t[:i] + (t[i] + t[i + 1],) + t[i + 2:]) + t[i] + t[i + 1] for i in range(len(t) - 1))\n    return go(tuple(piles))\ndef test_stones():\n    \"interval DP\"\n    assert merge_cost([4, 1, 3]) == 12 and merge_plan([4, 1, 3]) == \"(4 (1 3))\"\n    assert merge_cost([7]) == 0 and merge_plan([7]) == \"7\" and merge_cost([3, 4]) == 7\n    rng = random.Random(1)\n    for _ in range(60):\n        piles = [rng.randint(1, 9) for _ in range(rng.randint(1, 7))]\n        assert merge_cost(piles) == brute(piles)\n        plan = merge_plan(piles)\n        assert [int(x) for x in plan.replace(\"(\", \" \").replace(\")\", \" \").split()] == piles\n    big = [rng.randint(1, 100) for _ in range(120)]\n    assert merge_cost(big) > 0")

### Exercise: Lucky numbers

A number is **lucky** if its digits add up to a multiple of 7. Write `count_lucky(limit)`: how many numbers from 1 to `limit` are lucky, by digit DP with state (position, digit sum mod 7, tight). Then `count_lucky_between(lo, hi)`: how many in the range lo to hi inclusive.

In [ ]:
from functools import cache

def count_lucky(limit):
    ___

def count_lucky_between(lo, hi):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(lo, hi):\n    return sum(1 for x in range(lo, hi + 1) if sum(map(int, str(x))) % 7 == 0)\ndef test_lucky():\n    \"digit DP\"\n    assert count_lucky(100) == brute(1, 100) and count_lucky(6) == 0 and count_lucky(7) == 1 and count_lucky(0) == 0\n    rng = random.Random(2)\n    for _ in range(40):\n        lo = rng.randint(1, 3000)\n        hi = lo + rng.randint(0, 3000)\n        assert count_lucky_between(lo, hi) == brute(lo, hi)\n    assert abs(count_lucky(10 ** 18) - 10 ** 18 // 7) < 10 ** 16")

### Exercise: Dropping eggs

Write `fewest_drops(eggs, floors)` with the swapped DP (floors handled with t drops and k eggs), and `first_drop(eggs, floors)`: the floor to drop from first in an optimal strategy: one more than the floors that t − 1 drops and k − 1 eggs can handle below it (capped at the top floor), where t is the fewest drops.

In [ ]:
def fewest_drops(eggs, floors):
    ___

def first_drop(eggs, floors):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from functools import cache\ndef test_eggs():\n    \"eggs\"\n    @cache\n    def ref(k, n):\n        if n == 0:\n            return 0\n        if k == 1:\n            return n\n        return 1 + min(max(ref(k - 1, x - 1), ref(k, n - x)) for x in range(1, n + 1))\n    for k in range(1, 4):\n        for n in range(0, 40):\n            assert fewest_drops(k, n) == ref(k, n), (k, n)\n    assert fewest_drops(2, 100) == 14 and first_drop(2, 100) == 14 and first_drop(1, 10) == 1\n    assert fewest_drops(3, 10 ** 6) == 182 and fewest_drops(30, 10 ** 9) == 30\n    for k in range(1, 4):\n        for n in range(1, 40):\n            x = first_drop(k, n)\n            assert 1 <= x <= n and 1 + max(ref(k - 1, x - 1) if k > 1 else (x - 1), ref(k, n - x)) == ref(k, n)")

### Exercise: Splitting a number

Write `partitions(n)`: the number of ways to write n as a sum of positive whole numbers, ignoring order (5 = 4 + 1 = 3 + 2 = 3 + 1 + 1 = ...: 7 ways). Choose the state yourself (hint: it's an unbounded counting knapsack). Then `partitions_distinct(n)`: the ways using **different** parts only (a 0/1 counting knapsack).

In [ ]:
def partitions(n):
    ___

def partitions_distinct(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_partitions():\n    \"partitions\"\n    assert [partitions(n) for n in range(11)] == [1, 1, 2, 3, 5, 7, 11, 15, 22, 30, 42]\n    assert [partitions_distinct(n) for n in range(11)] == [1, 1, 1, 2, 2, 3, 4, 5, 6, 8, 10]\n    assert partitions(100) == 190569292 and partitions_distinct(100) == 444793")

### Exercise: The longest shared stretch

Unlike a subsequence, a **substring** has no gaps. Write `longest_common_substring(a, b)`: one longest string that is a substring of both (the first one found scanning a's end positions in order, on a tie). The state is "the longest common stretch **ending** at a[i − 1] and b[j − 1]": best ending here, in two dimensions, with two rows.

In [ ]:
def longest_common_substring(a, b):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef brute(a, b):\n    best = \"\"\n    for i in range(len(a)):\n        for j in range(i + 1, len(a) + 1):\n            if a[i:j] in b and j - i > len(best):\n                best = a[i:j]\n    return best\ndef test_substring():\n    \"contiguous\"\n    assert longest_common_substring(\"bananas\", \"panama\") == \"ana\"\n    assert longest_common_substring(\"abc\", \"xyz\") == \"\" and longest_common_substring(\"\", \"a\") == \"\"\n    rng = random.Random(4)\n    for _ in range(150):\n        a = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 9)))\n        b = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 9)))\n        s = longest_common_substring(a, b)\n        assert len(s) == len(brute(a, b)) and s in a and s in b")

### Bug hunt: Filling by start, not length

This interval DP fills `cost[i][j]` with i going **upwards**, but `cost[k + 1][j]` has a larger start, so it isn't filled yet, and it prints `8` instead of `12` for the piles `[4, 1, 3]`. Fill by increasing length (or i downwards).

In [ ]:
piles = [4, 1, 3]
n = len(piles)
prefix = [0]
for p in piles:
    prefix.append(prefix[-1] + p)
cost = [[0] * n for _ in range(n)]
for i in range(n):
    for j in range(i + 1, n):
        cost[i][j] = min(cost[i][k] + cost[k + 1][j] for k in range(i, j)) + prefix[j + 1] - prefix[i]
print(cost[0][n - 1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"by length\"\n    expect_output(\"12\")")

### Bug hunt: Always tight

This digit DP counts the numbers up to 35 whose digits add to 9, but it passes `tight` on unchanged, so even when the first digit is below 3 the next digit is still limited to 5. It prints `0` instead of `3` (9, 18 and 27). Stay tight only while each digit equals the limit's.

In [ ]:
from functools import cache

digits = [3, 5]

@cache
def go(pos, total, tight):
    if pos == len(digits):
        return 1 if total == 9 else 0
    top = digits[pos] if tight else 9
    return sum(go(pos + 1, total + d, tight) for d in range(top + 1))

print(go(0, 0, True))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"tight\"\n    expect_output(\"3\")")

### Bug hunt: A substring with gaps

For a common **substring**, a mismatch must reset the run to 0. This takes the larger neighbour instead (the LCS rule), so it finds a subsequence: it prints `3` for `"abcd"` and `"acbd"`, whose longest common substring has length `1`. Reset on a mismatch.

In [ ]:
a, b = "abcd", "acbd"
best = 0
prev = [0] * (len(b) + 1)
for i in range(1, len(a) + 1):
    cur = [0] * (len(b) + 1)
    for j in range(1, len(b) + 1):
        if a[i - 1] == b[j - 1]:
            cur[j] = prev[j - 1] + 1
        else:
            cur[j] = max(prev[j], cur[j - 1])
        best = max(best, cur[j])
    prev = cur
print(best)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"contiguous\"\n    expect_output(\"1\")")

### Puzzle: Interval DP

Arrange the lines of the matrix-chain DP's loops. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the fewest multiplications for matrices 10 × 30, 30 × 5, 5 × 60
q = ___   # fewest_drops(1, 10)
r = ___   # partitions(4)
s = ___   # the state pattern for "the fewest coins": "line", "grid", "interval" or "bitmask"
t = ___   # the work of an interval DP with n split points per stretch, as "O(n^2)" or "O(n^3)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 4500 and q == 10 and r == 5, \"p, q and r\"\n    assert s == \"line\" and t == \"O(n^3)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: bracketing a chain

Write `best_bracketing(dims)` for a chain of matrices where matrix i is `dims[i]` × `dims[i + 1]`: `(cost, brackets)`, the fewest scalar multiplications and the bracketing as a string over the names `A`, `B`, `C`, ... (one matrix is just its letter; two or more are wrapped in brackets with no spaces, like `"((AB)C)"`), keeping the **first** best split. Then `multiply_chain(matrices, brackets)`: actually multiply a list of matrices (lists of lists) in the order the brackets say, and return `(product, multiplications)`, counting the scalar multiplications, so the checks can confirm the cost is real. Raise `ValueError` if neighbouring shapes don't match.

In [ ]:
def best_bracketing(dims):
    ___

def multiply_chain(matrices, brackets):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, itertools\ndef all_costs(dims):\n    n = len(dims) - 1\n    from functools import cache\n    @cache\n    def go(i, j):\n        if i == j:\n            return 0\n        return min(go(i, k) + go(k + 1, j) + dims[i] * dims[k + 1] * dims[j + 1] for k in range(i, j))\n    return go(0, n - 1)\ndef test_chain():\n    \"matrix chains\"\n    assert best_bracketing([10, 30, 5, 60]) == (4500, \"((AB)C)\")\n    assert best_bracketing([40, 20, 30, 10, 30]) == (26000, \"((A(BC))D)\")\n    assert best_bracketing([5, 7]) == (0, \"A\")\n    rng = random.Random(5)\n    for _ in range(40):\n        dims = [rng.randint(1, 6) for _ in range(rng.randint(2, 6))]\n        cost, brackets = best_bracketing(dims)\n        assert cost == all_costs(dims)\n        mats = [[[rng.randint(0, 3) for _ in range(dims[i + 1])] for _ in range(dims[i])] for i in range(len(dims) - 1)]\n        product, count = multiply_chain(mats, brackets)\n        assert count == cost\n        left_to_right = mats[0]\n        for m in mats[1:]:\n            left_to_right = [[sum(r[k] * m[k][c] for k in range(len(m))) for c in range(len(m[0]))] for r in left_to_right]\n        assert product == left_to_right\n    try:\n        multiply_chain([[[1, 2]], [[1, 2]]], \"(AB)\")\n        assert False\n    except ValueError:\n        pass")

## Warm-up

Merging neighbouring piles of stones, paying the size of the merged pile each time: the cheapest way to merge piles i to j splits them at some k, merges each side, then joins the two. The table fills by **length** of the stretch, short to long, because a stretch depends only on shorter ones.

## The checklist

For any problem that smells of DP (Week 53 Day 1's signs):

1. **What's the question, as a function?** "f(...) = the best/number/possible for ...". Find the smallest set of arguments that fixes the rest of the answer. Try: a position (line), a pair of positions (two strings, a grid, a stretch), a position and a budget (knapsack), a node (tree, DAG), a set (bitmask), plus any small extra fact (holding, last colour, sideways moves used).
2. **What's the last (or first) decision?** List the options and what each leaves behind.
3. **Base cases** and **the answer's state**.
4. **Order**: what does each state read? Choose a loop order, or memoise.
5. **Count**: states × work per state. Too slow? Shrink the state, precompute something (prefix sums, a palindrome table), or use a better structure (bisect, a heap).
6. **Reconstruct** if the choices are wanted: keep the full table or a parent table.

## Interval DP

When the problem is about a **stretch** of a sequence that gets combined, split or removed from the inside (merging piles, multiplying a chain of matrices, bracketing an expression, removing balloons, cutting a stick), the state is the stretch:

**f(i, j) = the best for the stretch from i to j**,

and the choice is the **split point** k between i and j (or the element handled last). Each stretch reads strictly shorter ones, so fill by increasing length. There are O(n²) stretches and O(n) split points each: O(n³), fine for a few hundred elements.

The classic is **matrix chain multiplication**: multiplying an a × b matrix by a b × c one costs a × b × c steps, and the bracketing of a chain changes the total enormously, though not the result. (It appears in many textbooks; the O(n³) DP is due to Godbole, 1973.)

## Digit DP

"How many numbers from 0 to N have some property of their digits?" N may be 10¹⁸, so counting one by one is out. Build the numbers digit by digit from the left, with the state

**(position, some small fact about the digits so far, tight)**,

where **tight** says whether the digits so far equal N's (so the next digit can't exceed N's digit) or are already smaller (so any digit 0 to 9 is allowed). The small fact is whatever the property needs: the digit sum so far, the last digit, the remainder mod 7. For 19 digits with sums up to 171 and two values of tight, that's a few thousand states.

## Swap the question

Sometimes the obvious state is too big, but swapping "what you're given" with "what you compute" makes it small. The **egg drop** puzzle: with k eggs and a building of n floors, find the fewest drops that guarantee finding the highest safe floor (an egg breaks from any floor above it). The direct DP, f(k, n) = 1 + min over the floor x of max(f(k − 1, x − 1), f(k, n − x)), is O(k × n²). Swap it:

**floors(t, k) = the most floors you can handle with t drops and k eggs**.

Drop once: if it breaks, the floors below are handled with t − 1 drops and k − 1 eggs; if not, the floors above with t − 1 drops and k eggs. So floors(t, k) = floors(t − 1, k − 1) + 1 + floors(t − 1, k). Increase t until floors(t, k) ≥ n: O(k × answer), and the answer is at most n (and only about log n with plenty of eggs).

The same move appeared in Week 53 Day 5's knapsack by value, and Week 53 Day 6's least fee for a target crowd.

## Which pattern?

| The question involves | State | Week |
|---|---|---|
| a sequence, a contiguous run | i, ending here | 53 |
| cutting a sequence into pieces | prefix length i | 53 |
| a grid | (r, c) | 53 |
| a budget | (item, budget) | 53 |
| two strings | (i, j) prefixes | 54 |
| a stretch combined from inside | (i, j) stretch | 54 |
| a tree | node, small extras | 54 |
| a DAG | node in topological order | 54 |
| a small set, "each exactly once" | bitmask | 54 |
| numbers up to a huge limit | (digit, fact, tight) | 54 |

## Practice

## Quiz

## Daily challenge

## Before you go

Take any two problems from this week and run the checklist on paper for each, without looking at your code. Tomorrow is the Forge: a document comparison tool and a delivery-drone planner.